**Tabela** | ecommerce_categorias |

**Origem** | squad2/bronze/ecommerce_categorias (Delta) |

**Destino** | squad2/silver/ecommerce_categorias (Delta) |

**Modo** | Delta Streaming (Incremental via Checkpoint) |

**Regra Aplicada** | Regra Técnica 1: id_categoria e nome_categoria são obrigatórios |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA = "ecommerce_categorias"

# Caminhos ABFSS oficiais para o Delta Lake
path_bronze = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/{TABELA}"
path_silver = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"

# Caminho do arquivo de controle dentro do container
path_control = f"silver/control/{TABELA}.json"

print(f" Lendo de (Bronze): {path_bronze}")
print(f" Gravando em (Silver): {path_silver}")

try:
    # 1. Abre a tabela Bronze e converte para Pandas
    dt_bronze = DeltaTable(path_bronze, storage_options=get_storage_options())
    df_pandas = dt_bronze.to_pandas()
    
    # 2. CONTROLE INCREMENTAL: Instancia o cliente do arquivo de controle
    squad2_client = get_squad2_client()
    file_client = squad2_client.get_file_client(path_control)
    
    processados = set()
    
    # Se o arquivo JSON já existir na pasta silver/control, baixa e lê o conteúdo
    if file_client.exists():
        conteudo = file_client.download_file().readall().decode('utf-8')
        processados = set(json.loads(conteudo))
    
    # Filtra apenas os dados de arquivos que a Silver ainda não processou
    df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
    
    if df_novos_dados.empty:
        print(" Camada Silver em dia! Nenhum dado novo para processar.")
    else:
        print(f" Processando {len(df_novos_dados)} novas linhas da Bronze...")
        
        # 3. APLICAÇÃO DA REGRA TÉCNICA (id_categoria e nome_categoria obrigatórios)
        df_filtrado = df_novos_dados[
            df_novos_dados['id_categoria'].notna() & 
            df_novos_dados['nome_categoria'].notna() &
            (df_novos_dados['id_categoria'] != "") & 
            (df_novos_dados['nome_categoria'] != "")
        ].copy()
        
        # 4. COLUNA DE AUDITORIA
        df_filtrado['silver_processed_at'] = datetime.now()
        
        # Remove os fuso-horários para o formato Delta
        for col in df_filtrado.columns:
            if pd.api.types.is_datetime64_any_dtype(df_filtrado[col]):
                df_filtrado[col] = df_filtrado[col].dt.tz_localize(None)
                
        # 5. GRAVAÇÃO NA SILVER VIA DELTA
        write_deltalake(
            table_or_uri    = path_silver,
            data            = df_filtrado,
            mode            = "append",
            storage_options = get_storage_options()
        )
        
        # 6. ATUALIZA O CONTROL JSON USANDO O FILE CLIENT
        arquivos_atuais = set(df_novos_dados['bronze_source_file'].unique())
        todos_processados = list(processados.union(arquivos_atuais))
        
        # upload_data com overwrite=True cria o arquivo ou atualiza se já existir
        file_client.upload_data(json.dumps(todos_processados), overwrite=True)
        
        print(f" SUCESSO! {len(df_filtrado)} linhas filtradas e salvas com a control atualizada no Azure!")

except Exception as e:
    print(f" Erro no processamento: {str(e)}")
    raise